# Mathematical Reasoning: Formal Methods and LLM Integration

Mathematical reasoning represents the hardest end of LLM capability. Formal proof systems like Lean 4 provide ground-truth verification unavailable in natural language tasks. The integration of LLMs with formal mathematics is one of the most active research frontiers.

## The Spectrum of Math Benchmarks

**GSM8K** (grade school math): 8,500 word problems. Best models achieve 95%+. Essentially solved.

**MATH** (competition math): 12,500 competition problems, 5 difficulty levels. Hard — top models reach ~90% on easy, ~50% on the hardest problems.

**AIME** (American Invitational Mathematics Examination): 30 problems per year, integer answers 0-999. Requires deep mathematical insight. o1 achieves ~83%, humans average ~30%.

**IMO** (International Mathematical Olympiad): 6 problems requiring proofs. AlphaProof solved 4 of 6 in 2024.

**FrontierMath**: unpublished problems from working mathematicians. Specifically designed to prevent contamination. Top models achieve <2% without tools.

The progression shows that LLMs have nearly solved standard math benchmarks but are far from frontier mathematical research.

In [ ]:
from dataclasses import dataclass
from typing import Callable, Optional
import re

@dataclass
class MathProblem:
    problem: str
    answer: str
    difficulty: int  # 1-5
    domain: str

@dataclass
class ProofStep:
    tactic: str
    state_after: str
    verified: bool

class LeanVerifier:
    def __init__(self):
        # Simulate Lean verification: checks tactic validity
        self.valid_tactics = {
            'intro', 'apply', 'exact', 'rfl', 'simp', 'ring',
            'linarith', 'omega', 'norm_num', 'decide', 'trivial'
        }

    def verify_step(self, tactic: str, goal: str) -> tuple:
        tactic_name = tactic.split()[0] if tactic else ''
        valid = tactic_name in self.valid_tactics
        if valid:
            new_goal = f'Subgoal after {tactic_name}' if goal else 'proof complete'
        else:
            new_goal = goal
        return valid, new_goal

    def verify_proof(self, steps: list, initial_goal: str) -> dict:
        current_goal = initial_goal
        verified_steps = []
        for tactic in steps:
            valid, new_goal = self.verify_step(tactic, current_goal)
            verified_steps.append(ProofStep(
                tactic=tactic, state_after=new_goal, verified=valid
            ))
            if valid:
                current_goal = new_goal
            else:
                return {'success': False, 'failed_at': tactic,
                        'steps': verified_steps, 'completed': False}
        return {'success': True, 'steps': verified_steps,
                'final_state': current_goal, 'completed': True}

verifier = LeanVerifier()
goal = 'prove: forall n : Nat, n + 0 = n'
proof_steps = ['intro n', 'simp', 'ring']
result = verifier.verify_proof(proof_steps, goal)
print(f'Goal: {goal}')
print(f'Proof success: {result["success"]}')
for step in result['steps']:
    status = 'OK' if step.verified else 'FAIL'
    print(f'  [{status}] {step.tactic} -> {step.state_after}')

## LLM to Lean: The Translation Pipeline

The standard pipeline for formal theorem proving:
1. Express the theorem in natural language
2. LLM generates candidate Lean 4 tactic proofs
3. Lean verifier checks each proof
4. Failed proofs are fed back to the LLM with error messages
5. Repeat until a valid proof is found or budget exhausted

This loop is similar to test-driven development: the formal system provides precise feedback that guides the search.

In [ ]:
def llm_to_lean_pipeline(
    theorem: str,
    lean_verifier: LeanVerifier,
    llm_fn: Callable,
    max_attempts: int = 5
) -> dict:
    attempts = []
    for attempt in range(max_attempts):
        # LLM generates candidate proof tactics
        if attempt == 0:
            prompt = f'Prove in Lean 4 tactics: {theorem}'
        else:
            last = attempts[-1]
            error = last.get('failed_at', 'unknown error')
            prompt = f'Previous attempt failed at: {error}. Revise proof for: {theorem}'
        tactic_list = llm_fn(prompt)
        result = lean_verifier.verify_proof(tactic_list, theorem)
        attempts.append({**result, 'attempt': attempt+1, 'tactics': tactic_list})
        if result['success']:
            break
    return {
        'theorem': theorem,
        'total_attempts': len(attempts),
        'success': attempts[-1]['success'],
        'successful_attempt': next((a['attempt'] for a in attempts if a['success']), None),
        'attempts': attempts,
    }

rng = random.Random(42)
def mock_lean_llm(prompt):
    # Early attempts fail, later ones succeed
    attempt_num = prompt.count('attempt failed') + 1
    if attempt_num < 2:
        return ['intro n', 'invalid_tactic', 'simp']
    return ['intro n', 'simp', 'ring']

result = llm_to_lean_pipeline(
    'forall n : Nat, n + 0 = n',
    LeanVerifier(),
    mock_lean_llm,
    max_attempts=4
)
print(f'Theorem: {result["theorem"]}')
print(f'Attempts: {result["total_attempts"]}')
print(f'Success: {result["success"]} (on attempt {result["successful_attempt"]})')

## Neural Theorem Proving Landscape

Key systems and approaches:

**GPT-f** (Polu & Han 2020): first demonstration of GPT models generating useful proof steps in formal mathematics.

**Hypertree Proof Search** (Lample et al. 2022): MCTS guided by a language model policy and value function, trained on a large corpus of Lean proofs.

**AlphaProof** (DeepMind 2024): reinforcement learning over Lean proof search; solved 4 of 6 IMO 2024 problems. Used self-play to generate training data.

**LLM+SAT/SMT**: for problems expressible in formal logic, use LLMs to generate candidate formalizations and SAT/SMT solvers for verification.

The field is converging on a hybrid paradigm: LLMs for hypothesis generation and tactic suggestion, formal systems for verification.

## What Formal Methods Enable

The deep lesson: formal verification changes the failure mode of AI systems from silent errors to caught errors. A formally verified proof is correct by definition. A model-generated natural language proof can be fluent, plausible, and wrong.

For high-stakes mathematical applications (circuit verification, safety proofs, financial models), this distinction is critical. The current frontier: using LLMs to lower the effort required to write formal proofs, making formal verification practical for more domains.